# OIR Microscopy: Visualization and Preliminary Nuclear Segmentation

## Install OIR Dependencies

In [ ]:
!pip install oirfile[all]

## Inspect Acquisition Metadata

In [ ]:
from pprint import pprint
import oirfile

with oirfile.OirFile("data.oir") as oir:

    print("\n=== CHANNELS ===")
    pprint(oir.channels)

    print("\n=== ATTRIBUTES ===")
    pprint(oir.attrs)

    print("\n=== ACQUISITION ===")
    print("Datetime:", oir.datetime)
    print("Producer:", oir.producer)
    print("Scanner:", oir.scanner)

    print("\n=== SPATIAL INFORMATION ===")
    print("Dimensions:", oir.dims)
    print("Coordinate scales:", oir.coord_scales)
    print("Coordinate units:", oir.coord_units)
    print("Coordinate offsets:", oir.coord_offsets)

    print("\n=== XML METADATA ===")
    xml = oir.xml_metadata
    print(str(xml)[:10000])

## Load the Stack and Visualize a DAPI Slice

In [ ]:
import oirfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

file_path = "data.oir"

with oirfile.OirFile(file_path) as oir:
    image_data = oir.asarray()
    # Optional: read pixel size metadata if needed for a physical scale bar
    # pixel_size_x = oir.series.axes['X'].pixel_size

# 1. Select your target frame (Z=[0,15], C=0)
Z = 5
raw_frame = image_data[Z, 0, :, :].astype(float)

# 2. Contrast Adjusting (Percentile Stretching)
# Clips the top 0.5% of noise and scales the lowest 1% to prevent a muddy background
vmin, vmax = np.percentile(raw_frame, [1, 99.5])
clipped_frame = np.clip(raw_frame, vmin, vmax)

# 3. Create a custom high-contrast lookup table (LUT)
# Classic scientific pseudo-coloring (e.g., Green for Actin/Macrophage, Cyan/Blue for Nuclei)
# Change 'green' to 'cyan', 'magenta', or 'inferno' depending on what stain C=0 is!
cmap_color = 'inferno'

# Build the layout
fig, ax = plt.subplots(figsize=(6, 6), facecolor='black')
ax.set_facecolor('black')

# Display with stretched contrast and vibrant pseudo-color
img = ax.imshow(clipped_frame, cmap=cmap_color, vmin=vmin, vmax=vmax)

# 4. Remove borders and ticks for a clean presentation
ax.axis('off')
fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

# Save a high-res presentation-ready copy
plt.savefig("fancy_macrophage.png", dpi=300, facecolor=fig.get_facecolor(), edgecolor='none')
plt.show()

## Review Image Dimensions and Acquisition Details

In [ ]:
import oirfile

file_path = "data.oir"

print("=" * 60)
print(f"ANALYZING MICROSCOPY DATA: {file_path}")
print("=" * 60)

with oirfile.OirFile(file_path) as oir:
    # 1. Structural Dimensions
    print("[IMAGE DIMENSIONS]")
    print(f"  • Shape (Z, C, Y, X)  : {oir.shape}")
    print(f"  • Data Type          : {oir.dtype}")
    print(f"  • Total Z-Slices      : {oir.shape[0]}")
    print(f"  • Total Channels      : {oir.shape[1]}")
    print(f"  • Resolution per slice: {oir.shape[2]} x {oir.shape[3]} pixels")
    print("-" * 60)

    # 2. Native File Metadata Attributes
    print("[HARDWARE & FILE STATE]")

    # Safely pull properties exposed natively by the oirfile block handle
    for attr in ['version', 'is_lsm', 'is_compressed', 'bmptime']:
        if hasattr(oir, attr):
            val = getattr(oir, attr)
            print(f"  • {attr.replace('_', ' ').title():<18}: {val}")

    # Print underlying series parameters if populated inside the block mapping
    if hasattr(oir, 'series') and oir.series:
        print("\n[SERIES PROPERTY TREE]")
        for i, s in enumerate(oir.series):
            print(f"  Series {i}:")
            if hasattr(s, 'axes'):
                print(f"    - Axes Layout     : {s.axes}")
            if hasattr(s, 'dtype'):
                print(f"    - Array Data Type : {s.dtype}")

print("=" * 60)
print("Analysis Complete.")

## Visualize Individual Channels

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# True black-background fluorescence colormaps
cmaps = [
    "inferno",
    LinearSegmentedColormap.from_list("fluor_green", ["black", "lime"]),
    LinearSegmentedColormap.from_list("fluor_cyan", ["black", "cyan"]),
    LinearSegmentedColormap.from_list("fluor_magenta", ["black", "magenta"])
]

Z = 5
fig, axes = plt.subplots(1, 4, figsize=(16, 4), facecolor="black")

for c, ax in enumerate(axes):
    frame = image_data[Z, c].astype(np.float32)
    vmin, vmax = np.percentile(frame, [1, 99.5])

    ax.imshow(frame, cmap=cmaps[c], vmin=vmin, vmax=vmax)
    ax.set_title(f"Channel {c} | Z={Z}", color="white")
    ax.set_facecolor("black")
    ax.axis("off")

plt.tight_layout()
plt.show()



## View Maximum-Intensity Projections

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for c, ax in enumerate(axes):
    projection = image_data[:, c, :, :].max(axis=0)

    vmin, vmax = np.percentile(projection, [1, 99.5])

    ax.imshow(projection, cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_title(f"Channel {c} | Z-max projection")
    ax.axis("off")

plt.tight_layout()
plt.show()


## Build the Multichannel Composite

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# image_data shape: (Z, C, Y, X)
# (16, 4, 1024, 1024)

# Maximum-intensity projection for each channel
mip = image_data.max(axis=0)  # (C, Y, X)

def normalize_channel(channel):
    """Contrast stretch to [0, 1] for visualization."""
    channel = channel.astype(np.float32)
    vmin, vmax = np.percentile(channel, [1, 99.5])

    if vmax <= vmin:
        return np.zeros_like(channel)

    return np.clip(
        (channel - vmin) / (vmax - vmin),
        0, 1
    )

# Assign channels to RGB
rgb = np.stack([
    normalize_channel(mip[0]),  # Red
    normalize_channel(mip[1]),  # Green
    normalize_channel(mip[2]),  # Blue
], axis=-1)

fig, ax = plt.subplots(figsize=(8, 8), facecolor="black")
ax.imshow(rgb)
ax.set_title(
    "Da Vinci | Multichannel Fluorescence Composite",
    color="white"
)
ax.axis("off")

plt.tight_layout()
plt.savefig(
    "microscopy_rgb_composite.png",
    dpi=300,
    facecolor="black",
    bbox_inches="tight"
)
plt.show()


# Nuclear Segmentation

## Environment Check and Cellpose Installation

In [ ]:
import sys
print(sys.version)

In [ ]:
%pip install --only-binary=numpy "numpy==2.0.2" "cellpose==3.1.1.1" matplotlib "oirfile[all]"

## Load the Stack for Segmentation

In [ ]:
from pathlib import Path
import oirfile

file_path = Path("/content/data.oir")

with oirfile.OirFile(file_path) as oir:
    axes = tuple(oir.dims)
    if axes != ("Z", "C", "Y", "X"):
        raise ValueError(f"Unexpected axis order: {axes}")

    image_data = oir.asarray()

print("Axes:", axes)
print("Shape:", image_data.shape)
print("Data type:", image_data.dtype)

## Confirm DAPI and Configure Segmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DAPI_CONFIRMED = True
DAPI_CONFIRMATION_SOURCE = (
    "data.oir acquisition XML: enabled CH1, order=1, "
    "channel id=f22975c3-8548-4676-8bb0-384e4bd47d7d, dyeName=DAPI"
)

DAPI_CHANNEL = 0  # CH1 in the (Z, C, Y, X) array
Z_INDEX = 5

NUCLEAR_DIAMETER_UM = 10.0  # initial estimate for tuning
XY_SPACING_UM = 0.310740284701119

if not DAPI_CONFIRMED or not DAPI_CONFIRMATION_SOURCE.strip():
    raise RuntimeError(
        "Confirm DAPI identity and channel mapping, then set "
        "DAPI_CONFIRMED=True and record DAPI_CONFIRMATION_SOURCE."
    )

if image_data.ndim != 4:
    raise ValueError(
        f"Expected verified (Z, C, Y, X) layout; got {image_data.shape}."
    )

if not 0 <= DAPI_CHANNEL < image_data.shape[1]:
    raise ValueError("DAPI_CHANNEL is outside the available channels.")

if not 0 <= Z_INDEX < image_data.shape[0]:
    raise ValueError("Z_INDEX is outside the available slices.")

if XY_SPACING_UM <= 0 or NUCLEAR_DIAMETER_UM <= 0:
    raise ValueError("Pixel spacing and nuclear diameter must be positive.")

# A separate float32 copy: image_data remains unchanged.
nuclear_image = image_data[Z_INDEX, DAPI_CHANNEL].astype(
    np.float32, copy=True
)

if not np.isfinite(nuclear_image).all():
    raise ValueError("Selected image contains non-finite values.")

if nuclear_image.max() <= nuclear_image.min():
    raise ValueError("Selected image is constant; choose another slice.")

diameter_px = NUCLEAR_DIAMETER_UM / XY_SPACING_UM
print(f"Selected Z={Z_INDEX}, channel index={DAPI_CHANNEL}")
print(f"Initial diameter: {diameter_px:.1f} pixels")

## Run Pretrained Cellpose

In [ ]:
from importlib.metadata import version
from cellpose import models

if version("cellpose") != "3.1.1.1":
    raise RuntimeError(
        "These cells require cellpose==3.1.1.1. "
        "Install that version and restart the kernel."
    )

model = models.CellposeModel(
    gpu=False,
    model_type="nuclei",
)

masks, flows, styles = model.eval(
    nuclear_image,
    channels=[0, 0],       # single grayscale image
    diameter=diameter_px,
    normalize=True,       # Cellpose handles inference normalization
    do_3D=False,
    flow_threshold=0.4,
    cellprob_threshold=0.0,
    min_size=15,           # pixels; starting value, not biologically validated
)

masks = np.asarray(masks)
if masks.shape != nuclear_image.shape:
    raise RuntimeError(f"Unexpected mask shape: {masks.shape}")

object_ids = np.unique(masks)
object_ids = object_ids[object_ids != 0]
print(f"Preliminary segmented objects in this slice: {len(object_ids)}")

## DAPI Segmentation Overlay

In [ ]:
from cellpose import utils

vmin, vmax = np.percentile(nuclear_image, [1, 99.5])
if vmax <= vmin:
    vmin, vmax = float(nuclear_image.min()), float(nuclear_image.max())

outlines = utils.masks_to_outlines(masks)

outline_rgba = np.zeros((*masks.shape, 4), dtype=np.float32)
outline_rgba[outlines] = (1.0, 0.2, 0.2, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax in axes:
    ax.imshow(nuclear_image, cmap="gray", vmin=vmin, vmax=vmax)
    ax.axis("off")

axes[0].set_title(f"Confirmed DAPI | Z={Z_INDEX}")
axes[1].imshow(outline_rgba)
axes[1].set_title(
    f"Preliminary Cellpose outlines | {len(object_ids)} objects"
)

plt.tight_layout()
plt.show()

if len(object_ids) == 0:
    print("No objects detected. Inspect channel, slice, contrast, and diameter.")

## Export Private Masks, Figures, and Parameters

In [ ]:
from pathlib import Path
import json
import platform

SAVE_LOCAL_RESULTS = True

if SAVE_LOCAL_RESULTS:
    output_dir = Path("private_results")
    output_dir.mkdir(exist_ok=True)

    stem = f"nuclei_z{Z_INDEX:03d}_c{DAPI_CHANNEL}"

    np.save(output_dir / f"{stem}_masks.npy", masks)
    fig.savefig(
        output_dir / f"{stem}_overlay.png",
        dpi=200,
        bbox_inches="tight",
    )

    metadata = {
        "python_version": platform.python_version(),
        "cellpose_version": version("cellpose"),
        "numpy_version": version("numpy"),
        "matplotlib_version": version("matplotlib"),
        "model": "nuclei",
        "dapi_confirmed": DAPI_CONFIRMED,
        "confirmation_source": DAPI_CONFIRMATION_SOURCE,
        "channel_index": DAPI_CHANNEL,
        "z_index": Z_INDEX,
        "xy_spacing_um": XY_SPACING_UM,
        "diameter_um_initial_estimate": NUCLEAR_DIAMETER_UM,
        "diameter_px": diameter_px,
        "flow_threshold": 0.4,
        "cellprob_threshold": 0.0,
        "min_size_px": 15,
        "object_count_2d": len(object_ids),
        "validated": False,
    }

    (output_dir / f"{stem}_parameters.json").write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )
    print(f"Saved locally in {output_dir.resolve()}")

## Prepare the Multichannel Projection for Overlay

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# image_data shape: (Z, C, Y, X)
# (16, 4, 1024, 1024)

# Maximum-intensity projection for each channel
mip = image_data.max(axis=0)  # (C, Y, X)

def normalize_channel(channel):
    """Contrast stretch to [0, 1] for visualization."""
    channel = channel.astype(np.float32)
    vmin, vmax = np.percentile(channel, [1, 99.5])

    if vmax <= vmin:
        return np.zeros_like(channel)

    return np.clip(
        (channel - vmin) / (vmax - vmin),
        0, 1
    )

# Assign channels to RGB
rgb = np.stack([
    normalize_channel(mip[0]),  # Red
    normalize_channel(mip[1]),  # Green
    normalize_channel(mip[2]),  # Blue
], axis=-1)

fig, ax = plt.subplots(figsize=(8, 8), facecolor="black")
ax.imshow(rgb)
ax.set_title(
    "Multichannel Fluorescence Composite",
    color="white"
)
ax.axis("off")

plt.tight_layout()
plt.savefig(
    "microscopy_rgb_composite.png",
    dpi=300,
    facecolor="black",
    bbox_inches="tight"
)
plt.show()


## Display the Composite Segmentation Overlay

In [ ]:
from cellpose import utils
from scipy.ndimage import binary_dilation

# Transparent layer with yellow segmentation boundaries
outlines = utils.masks_to_outlines(masks)

if masks.shape != rgb.shape[:2]:
    raise ValueError("Masks and composite must have the same XY dimensions.")

# Thicken boundaries for display; segmentation masks stay unchanged.
thick_outlines = binary_dilation(outlines, iterations=1)

# Wider black backing, narrower cyan line
backing = binary_dilation(outlines, iterations=2)
colored_line = binary_dilation(outlines, iterations=1)

overlay = np.zeros((*masks.shape, 4), dtype=np.float32)
overlay[backing] = (0, 0, 0, 1)
overlay[colored_line] = (232/255, 240/255, 1.0, 1.0)

fig, ax = plt.subplots(figsize=(10, 10), facecolor="black")
ax.imshow(rgb)
ax.imshow(overlay, interpolation="nearest")

ax.set_title(
    f"Multichannel Z-max projection\n"
    f"DAPI segmentation outlines from Z={Z_INDEX}",
    color="white",
    fontsize=12,
)
ax.axis("off")

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

figure_path = figure_dir / f"composite_nuclei_z{Z_INDEX:03d}.png"

fig.tight_layout()
fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor=fig.get_facecolor(),
    edgecolor="none",
)
plt.show()

print(f"Saved: {figure_path.resolve()}")